In [1]:
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

2.11.0+cu130 True Tesla T4


In [3]:
def bench(fn, iters=50, warmup=10):
    for _ in range(warmup): fn()
    torch.cuda.synchronize()
    s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    s.record()
    for _ in range(iters): fn()
    e.record(); torch.cuda.synchronize()
    return s.elapsed_time(e) / iters / 1000   # seconds per call

In [4]:
# 1) Big matmul: high arithmetic intensity (should be compute-bound)
N = 4096
a = torch.randn(N, N, device="cuda", dtype=torch.float16)
b = torch.randn(N, N, device="cuda", dtype=torch.float16)
t = bench(lambda: a @ b)
print(f"Matmul     : {2*N**3 / t / 1e12:.1f} TFLOPS")

Matmul     : 21.6 TFLOPS


In [5]:
# 2) Big elementwise add: very low arithmetic intensity (should be memory-bound)
n = 100_000_000
x = torch.randn(n, device="cuda", dtype=torch.float16)
y = torch.randn(n, device="cuda", dtype=torch.float16)
t = bench(lambda: x + y)
print(f"Vector add : {3*n*2 / t / 1e9:.0f} GB/s moved, {n / t / 1e12:.3f} TFLOPS")


Vector add : 252 GB/s moved, 0.042 TFLOPS


In [6]:
# 3) Many tiny ops: launch overhead dominates
small = torch.randn(1000, device="cuda")
def many_small():
    for _ in range(1000):
        small.add_(1)
t = bench(many_small, iters=5, warmup=2)
print(f"1000 tiny ops: {t*1000:.1f} ms total, {t/1000*1e6:.1f} microseconds per op")


1000 tiny ops: 7.8 ms total, 7.8 microseconds per op


In [7]:
from torch.profiler import profile, ProfilerActivity

with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
    a @ b
    x + y
    many_small()
    torch.cuda.synchronize()

print(prof.key_averages().table(sort_by="device_time_total", row_limit=10))
prof.export_chrome_trace("trace.json")

/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                           aten::matmul         0.17%      28.484us        24.75%       4.083ms       4.083ms       0.000us         0.00%      13.087ms      13.087ms             1  
                                               aten::mm         1.17%     192.605us        24.57%       4.055ms       4.055ms       6.543ms        52.47%      13.087ms      13.087ms             1  
         